In [1]:
%pip install -q -U markdown-pdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
!{sys.executable} -m pip install -q -U markdown-pdf

In [3]:
import json
import re
from pathlib import Path

import fitz
from markdown_pdf import MarkdownPdf, Section

In [10]:
APP_ROOT = Path("/home/dhruva/pop_gemini_direct_pdf_pipeline")

DOC_NAME = "kannada_test_job"
JOB_ROOT = APP_ROOT / "workdir" / DOC_NAME
PAGES_PDF_ROOT = JOB_ROOT / "pages_pdf"

START_PAGE = 16
END_PAGE = 20
OVERWRITE_EXISTING = False

print("Pages PDF root:", PAGES_PDF_ROOT)
print("Exists:", PAGES_PDF_ROOT.exists())

Pages PDF root: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf
Exists: True


In [11]:
DEFAULT_CSS = """
body {
  font-family: Arial, sans-serif;
  font-size: 11pt;
  line-height: 1.35;
}
table, th, td {
  border: 1px solid black;
  border-collapse: collapse;
}
th, td {
  padding: 4px;
  vertical-align: top;
}
h1, h2, h3, h4, h5, h6 {
  margin-top: 14px;
  margin-bottom: 8px;
}
pre, code {
  white-space: pre-wrap;
}
"""

In [12]:
def clean_markdown_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", text)
    text = re.sub(r"\n```$", "", text)
    return text.strip() + "\n"


def convert_markdown_file_to_pdf(md_path: Path, pdf_path: Path):
    text = md_path.read_text(encoding="utf-8")
    text = clean_markdown_text(text)

    pdf = MarkdownPdf(toc_level=0, optimize=True)
    pdf.meta["title"] = md_path.stem
    pdf.add_section(Section(text), user_css=DEFAULT_CSS)
    pdf.save(str(pdf_path))

In [14]:
summary = []

for page_num in range(START_PAGE, END_PAGE + 1):
    page_name = f"page_{page_num}"
    page_dir = PAGES_PDF_ROOT / page_name
    md_path = page_dir / "translated_en.md"
    pdf_path = page_dir / "translated_en.pdf"
    err_path = page_dir / "translated_en_pdf_error.txt"

    if not md_path.exists():
        msg = f"translated_en.md not found: {md_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        if page_dir.exists():
            err_path.write_text(msg, encoding="utf-8")
        continue

    if pdf_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {page_name}: translated_en.pdf already exists")
        summary.append({
            "page": page_name,
            "status": "skipped",
            "reason": "translated_en.pdf already exists",
        })
        continue

    print(f"Converting {page_name} ...")

    try:
        convert_markdown_file_to_pdf(md_path, pdf_path)
        summary.append({
            "page": page_name,
            "status": "success",
            "output_file": str(pdf_path),
        })
        print(f"Saved: {pdf_path}")

    except Exception as e:
        err = str(e)
        err_path.write_text(err, encoding="utf-8")
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": err,
            "error_file": str(err_path),
        })
        print(f"Failed: {page_name} -> {err}")

Converting page_16 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_16/translated_en.pdf
Converting page_17 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_17/translated_en.pdf
Converting page_18 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_18/translated_en.pdf
Converting page_19 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_19/translated_en.pdf
Converting page_20 ...
Saved: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_20/translated_en.pdf


In [8]:
summary_path = JOB_ROOT / "step3_page_pdf_generation_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"Saved summary: {summary_path}")
summary

Saved summary: /home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/step3_page_pdf_generation_summary.json


[{'page': 'page_16',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_16/translated_en.pdf'},
 {'page': 'page_17',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_17/translated_en.pdf'},
 {'page': 'page_18',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_18/translated_en.pdf'},
 {'page': 'page_19',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_19/translated_en.pdf'},
 {'page': 'page_20',
  'status': 'success',
  'output_file': '/home/dhruva/pop_gemini_direct_pdf_pipeline/workdir/kannada_test_job/pages_pdf/page_20/translated_en.pdf'}]